# Method 1: BERT Hybrid + XGBoost (混合特徵架構)

### 為什麼我們認為這個方法會更好？ (Why is it better?)
1. **解除 TF-IDF 稀疏魔咒**：我們不再將 5000 維、充滿 `0` 的稀疏矩陣直接餵給分類器。取而代之的是，我們利用前沿的 `Sentence-Transformer` 語言模型，直接將整句話轉為 **768 維的高濃度語意向量 (Dense Embeddings)**。
2. **保證領域感知**：我們依舊執行「雙軌制」，透過原本乾淨純粹的 TF-IDF 餵給 LDA，萃取出 **3 維的 Soft Topics**（告訴模型這句話屬於電影、商品還是遊戲領域），以及 **2 維的 Meta 特徵** (驚嘆號、問號頻率)。
3. **換上非線性王者**：當我們的特徵從「稀疏的 5000 維字彙表」變成了「總計約 773 維」充滿語意、機率與次數的**連續密集特徵**時，傳統的 Logistic Regression 將無法捕捉這些深層關係。我們將預測引擎換成樹狀模型霸主 **XGBoost**，讓它自由去切割這些極度純粹的數值，這將徹底碾壓 Baseline！

In [1]:
import pandas as pd
import numpy as np
import re
import warnings

# Sklearn (Topic, ML tools)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import cross_val_score

# Deep Learning Embbedding
from sentence_transformers import SentenceTransformer

# 最終樹狀預測模型
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# 載入資料 (我們維持在 EDA/ 目錄，所以回上一層 ../datasets)
train_df = pd.read_csv('../datasets/train_2022.csv')
test_df = pd.read_csv('../datasets/test_2022.csv')
sample_sub = pd.read_csv('../datasets/sample_submission.csv')

print(f"Train Shape: {train_df.shape}")
print(f"Test Shape: {test_df.shape}")

Train Shape: (2000, 3)
Test Shape: (11000, 2)


In [2]:
# 1. 前處理機制
# 【軌道 A】用來提取 TF-IDF （只保留字母） -> 後送給 LDA
def clean_for_tfidf(text):
    if not isinstance(text, str):
        return ""
    text = str(text).lower()
    text = text.replace('-lrb-', ' ').replace('-rrb-', ' ')
    text = re.sub(r'[^a-zA-Z]', ' ', text) # 只保留純英文字母
    return re.sub(r'\s+', ' ', text).strip()

# 【軌道 B】用來給 BERT 萃取句子的原汁原味感情 (還原括號等標籤)
def clean_for_bert(text):
    if not isinstance(text, str):
        return ""
    text = text.replace('-lrb-', '(').replace('-rrb-', ')')
    return text.strip()

# 【特徵】萃取 Meta 特徵
def extract_meta_features(df):
    meta_df = pd.DataFrame()
    meta_df['q_mark_count'] = df['TEXT'].apply(lambda x: str(x).count('?'))
    meta_df['e_mark_count'] = df['TEXT'].apply(lambda x: str(x).count('!'))
    return meta_df.values # 直接返回 Numpy 陣列

# 建立兩個軌道版本的文字
print(">> 開始兵分兩路之前處理...")
train_text_tfidf = train_df['TEXT'].apply(clean_for_tfidf)
test_text_tfidf = test_df['TEXT'].apply(clean_for_tfidf)

train_text_bert = train_df['TEXT'].apply(clean_for_bert)
test_text_bert = test_df['TEXT'].apply(clean_for_bert)

print(">> 完成資料前處理！")

>> 開始兵分兩路之前處理...
>> 完成資料前處理！


In [3]:
# 2. 獲取 Topic 特徵 (3維 Dense) 與 Meta 特徵 (2維 Dense)
print(">> 開始建立 TF-IDF 矩陣並萃取 LDA 軟性主題...")

# 準備 TF-IDF 矩前
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', min_df=2)
# 我們需要它，但只有為了餵給 LDA 而已！
train_tfidf_matrix = tfidf.fit_transform(train_text_tfidf)
test_tfidf_matrix = tfidf.transform(test_text_tfidf)

# 訓練 LDA (獲得 3維 機率特徵)
n_topics = 3
lda_model = LatentDirichletAllocation(n_components=n_topics, random_state=42)

# X_train_topics: (2000, 3) 機率數字 Dense Numpy Array
X_train_topics = lda_model.fit_transform(train_tfidf_matrix)
X_test_topics = lda_model.transform(test_tfidf_matrix)

print(f"訓練集 Soft Topic 形狀: {X_train_topics.shape}")

# 獲取 Meta
train_meta_features = extract_meta_features(train_df)
test_meta_features = extract_meta_features(test_df)
print(f"訓練集 Meta 形狀: {train_meta_features.shape}")

>> 開始建立 TF-IDF 矩陣並萃取 LDA 軟性主題...
訓練集 Soft Topic 形狀: (2000, 3)
訓練集 Meta 形狀: (2000, 2)


In [4]:
# 3. 獲取強大的 BERT Embeddings
print(">> 下載並設定 Sentence Transformer 模型 (all-MiniLM-L6-v2) ...")

# 這裡我們使用輕量且快速的模型版本 (跑純 CPU 也很快)，如果機器跑得慢也相對寬容。
model_name = 'all-MiniLM-L6-v2'
st_model = SentenceTransformer(model_name)

# 透過 Track B 原汁原味的標點符號跟上下文供語意萃取
print(">> 正在為 Train Set 生成 Embeddings (這可能需要 1~2 分鐘)...")
X_train_bert = st_model.encode(train_text_bert.tolist(), batch_size=32, show_progress_bar=True)

print(">> 正在為 Test Set 生成 Embeddings...")
X_test_bert = st_model.encode(test_text_bert.tolist(), batch_size=32, show_progress_bar=True)

print(f"訓練集 BERT Embeddings 形狀: {X_train_bert.shape} (這是 {X_train_bert.shape[1]} 維的 Dense 矩陣！)")

>> 下載並設定 Sentence Transformer 模型 (all-MiniLM-L6-v2) ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

>> 正在為 Train Set 生成 Embeddings (這可能需要 1~2 分鐘)...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

>> 正在為 Test Set 生成 Embeddings...


Batches:   0%|          | 0/344 [00:00<?, ?it/s]

訓練集 BERT Embeddings 形狀: (2000, 384) (這是 384 維的 Dense 矩陣！)


In [5]:
# 4. 特徵總拼接 (Feature Concatenation) + Numpy 高效整合!
# 因為現在沒有任何 Scipy Sparse Matrix 稀疏特徵參與，三個全都是連續數字，我們可以直接使用 np.hstack!
print(">> 將 BERT 跟 Meta 與 LDA 主題合併起來，組成究極體 Dense Matrix !")
X_train_final = np.hstack([X_train_bert, X_train_topics, train_meta_features])
X_test_final = np.hstack([X_test_bert, X_test_topics, test_meta_features])
y_train = train_df['LABEL'].values

print("最後合併的訓練集特徵維度: ", X_train_final.shape)
print("  包含: [BERT 384] + [LDA 主題機率 3] + [Meta 標點 2] = 389 終極維度")
# (註：MiniLM 預設為 384 維度, 如果用 bert-base 是 768)

>> 將 BERT 跟 Meta 與 LDA 主題合併起來，組成究極體 Dense Matrix !
最後合併的訓練集特徵維度:  (2000, 389)
  包含: [BERT 384] + [LDA 主題機率 3] + [Meta 標點 2] = 389 終極維度


In [6]:
# 5. 上機！XGBoost 分類器模型與交叉驗證 (Cross Validation)
# XGBoost 在全 Dense 連續變數 (不超過 500 維) 的樹狀切分威力驚人！
print("====== [ 訓練模型: XGBoost Classifier ] ======")
xgb = XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=4, 
    random_state=42, 
    n_jobs=-1,        # 使用全部 CPU 核心  
    use_label_encoder=False, 
    eval_metric='logloss'
)

# 進行 5 折交叉驗證來偷看模型準確度！(Baseline LR 約為 64.85%)
cv_scores = cross_val_score(xgb, X_train_final, y_train, cv=5, scoring='accuracy')

print(f"🌲 XGBoost 5-Fold 分數: {cv_scores}")
print(f"⭐⭐ XGBoost 平均準確率: {cv_scores.mean():.4f}")

# 6. 開始預測 Test 並輸出
print(">> 開始訓練總模型，以及準備生出預測結果給 Test...")
xgb.fit(X_train_final, y_train)
test_preds = xgb.predict(X_test_final)

# DataFrame 組裝 Kaggle 標準答案
submission = pd.DataFrame({
    'row_id': test_df['row_id'],
    'LABEL': test_preds
})

submission.to_csv('../submission_method1_xgb.csv', index=False)
print("✅ 模型訓練完成！以儲存在 '../submission_method1_xgb.csv'")

====== [ 訓練模型: XGBoost Classifier ] ======
🌲 XGBoost 5-Fold 分數: [0.68   0.69   0.74   0.7275 0.725 ]
⭐⭐ XGBoost 平均準確率: 0.7125
>> 開始訓練總模型，以及準備生出預測結果給 Test...
✅ 模型訓練完成！以儲存在 '../submission_method1_xgb.csv'
